# Judge Worker

Consumes `unscored/` milestone snapshots written by `edit.ipynb`, writes `scored/`, and rebuilds `perf/` result files for visualization. Run this in a separate kernel/process while the edit producer is running.


In [ ]:
import sys
from datetime import datetime
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "src").exists() and (path / "notebook").exists():
            return path
    raise RuntimeError(f"Could not find project root from {start}")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

LOG_DIR = PROJECT_ROOT / "data" / "evaluation" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / f"judge-{datetime.now():%Y-%m-%d-%H-%M-%S}.log"


In [ ]:
import copy

import yaml


EVAL_CONFIG_PATH = PROJECT_ROOT / "src" / "evaluate" / "config.yaml"
with open(EVAL_CONFIG_PATH, encoding="utf-8") as file:
    eval_config = yaml.safe_load(file)

editing_methods = list(eval_config["editing_methods"])
model_name = eval_config["model_name"]
result_names = list(eval_config["result_names"])
eval_milestones = list(eval_config["eval_milestones"])
n_runs = int(eval_config["n_runs"])
judge_repeats = int(eval_config["judge_repeats"])
base_seed = int(eval_config["base_seed"])

judge_config = copy.deepcopy(eval_config["judge"])
judge_config["repeats"] = judge_repeats
judge_config["log_path"] = LOG_PATH
judge_config.setdefault("model_config", {})["seed"] = base_seed

worker_config = copy.deepcopy(eval_config["worker"])


In [ ]:
from src.evaluate.judge_worker import score_pending_unscored

await score_pending_unscored(
    model=model_name,
    milestones=eval_milestones,
    n_runs=n_runs,
    judge_repeats=judge_repeats,
    judge_config=judge_config,
    methods=editing_methods,
    result_names=result_names,
    **worker_config,
)


## Notes

- `file_concurrency` controls how many milestone files are scored at once.
- `judge_config["max_concurrency"]` controls how many judge API requests each file can launch internally.
- Total judge pressure is roughly `file_concurrency * judge_config["max_concurrency"]`.
- The worker is idempotent: existing `scored/` files are skipped unless `overwrite=True`, and `perf/` can be rebuilt from existing `scored/` files.
